# 04 — Statistical Hypothesis Testing

**Football Odds Intelligence & Match Analytics**

---

## Hypotheses Formulated
1. **Bookmaker Margin Disparity ($H_1$):** Do Pinnacle, Bet365, SBOBET, and Singbet operate with statistically distinct pricing margins?
   - **Test:** Kruskal-Wallis H-test & Mann-Whitney U test.
2. **Systematic Odds Drift ($H_2$):** Do odds systematically drift away from opening quotes by kickoff?
   - **Test:** Wilcoxon Signed-Rank Test.
3. **Competition Independence ($H_3$):** Is the distribution of match outcomes (Home, Draw, Away) independent of the league?
   - **Test:** Pearson Chi-Square Test of Independence.
4. **Home Advantage Significance ($H_4$):** Is the global Home Win rate significantly higher than the random baseline ($33.33\%$)?
   - **Test:** One-sample Z-test for proportions.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_processed
from src.evaluation.statistical_tests import (
    evaluate_bookmaker_margin_differences,
    evaluate_odds_drift_significance,
    evaluate_league_outcome_independence,
    evaluate_home_advantage_significance
)

matches = load_processed('matches_clean')
closing = load_processed('match_closing_clean')

## 1. Bookmaker Margins (Kruskal-Wallis & Mann-Whitney)

In [ ]:
margin_results = evaluate_bookmaker_margin_differences(closing)
print(f"H-Statistic: {margin_results['h_statistic']:.2f}, p-value: {margin_results['p_value']:.4e}")
print(f"Pinnacle vs Bet365 Mann-Whitney p-value: {margin_results['pinnacle_vs_bet365_p_value']:.4e}")
pd.DataFrame(margin_results['bookmaker_summaries']).T

## 2. Odds Drift (Wilcoxon Signed-Rank Test)

In [ ]:
drift_results = evaluate_odds_drift_significance(closing)
pd.DataFrame(drift_results).T

## 3. League vs Outcome Independence (Chi-Square Test)

In [ ]:
league_results = evaluate_league_outcome_independence(matches, min_matches=200)
print(f"Chi-Square Statistic: {league_results['chi2_statistic']:.2f}")
print(f"Degrees of Freedom:   {league_results['degrees_of_freedom']}")
print(f"p-value:              {league_results['p_value']:.4e}")
print(f"Significant:          {league_results['is_significant']}")

## 4. Home Advantage (Proportion Z-Test)

In [ ]:
home_results = evaluate_home_advantage_significance(matches)
print(f"Total Matches Evaluated: {home_results['total_matches']:,}")
print(f"Home Win Rate:           {home_results['home_win_rate']*100:.2f}%")
print(f"Z-statistic:             {home_results['z_statistic']:.2f} (p = {home_results['p_value']:.4e})")
print(f"95% Confidence Interval: [{home_results['ci_95'][0]*100:.2f}%, {home_results['ci_95'][1]*100:.2f}%]")